# Sentencing Factor Labeling Lite: demo (Colab)

Runs the labeling pipeline of the paper on synthetic demo data: chunking, chunk utility assessment (CUA), factor / chunk similarity (BGE-M3 by default), a threshold, and the metrics. Optionally it also fine-tunes the encoder.

- **Section 2** runs the pipeline (no training).
- **Section 3** adds the fine-tuning step (needs a GPU).
- **Section 4** is an optional cross-validated evaluation; **Section 5** has checks that need no model download.

Use a GPU runtime (Runtime > Change runtime type); the first run downloads BGE-M3. The demo data is synthetic, so the scores show the pipeline running end to end.

## 1. Get the code

The next cell clones the repository from GitHub and installs it.

In [ ]:
%cd /content
!git clone -q https://github.com/tedlctai/sentencing-factor-labeling-lite.git || echo "already cloned"
%cd /content/sentencing-factor-labeling-lite
!pip -q install -e . sentence-transformers

## 2. Run the labeling pipeline

No training. The table has the layout of the paper's Table II (accuracy, precision, recall, F1, Hamming loss) for the three variants: whole text, chunking, and chunking with CUA.

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python scripts/run_demo.py
print(open('outputs/demo/results.md', encoding='utf-8').read())

## 3. Fine-tune the encoder (optional, needs a GPU)

One command: it builds the fine-tuning data from the 640 judgments reserved in `splits/sft.json`, fine-tunes the encoder once, and adds three SFT rows to the table. The other 160 judgments are evaluated, as in Section 2. On the demo data a mock LLM verifier stands in for the LLM, so no key is needed. The settings in `configs/demo_train.yaml` are shrunk to fit one Colab GPU; the paper's values are in `configs/paper.yaml`.

In [ ]:
!python scripts/run_demo.py --finetune --out outputs/demo_finetune
print(open('outputs/demo_finetune/results.md', encoding='utf-8').read())

**With a real LLM (optional).** Add your key as a Colab secret named `OPENAI_API_KEY` (any OpenAI-compatible endpoint works) and give the model name. One short request is sent per (factor, top-k chunk); answers are cached, so a re-run costs nothing, and `build_sft_data.py --dry-run` shows the count first (docs/implementation_notes.md). To run the steps one at a time, see the README.

In [ ]:
import os
from google.colab import userdata
LLM_MODEL = 'YOUR_MODEL_NAME'      # the model that answers Y/N for each (factor, chunk)
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
!python scripts/run_demo.py --finetune --llm-model {LLM_MODEL} --out outputs/demo_finetune_llm
print(open('outputs/demo_finetune_llm/results.md', encoding='utf-8').read())

## 4. Optional: cross-validated evaluation

Use this when there is no separate test set and every judgment should still be evaluated by an encoder that did not see it. The CUA is fitted once and reused; for each of 5 folds it builds the fine-tuning data from the other folds, fine-tunes an encoder and scores the held-out fold, and the threshold is chosen on the other folds. This trains 5 times, so it needs a GPU runtime.

In [ ]:
!python scripts/run_demo.py --crossval --out outputs/crossval
print(open('outputs/crossval/results.md', encoding='utf-8').read())

## 5. Optional: checks without a model download

The unit tests and the hash encoder only check that the scripts run; the hash encoder is only for these smoke tests.

In [ ]:
!pip -q install pytest
!python -m unittest discover -s tests
!python scripts/run_demo.py --encoder hash --taus 0.10,0.15,0.20,0.25,0.30,0.35 --out outputs/demo_hash